# Data Preprocessing Pipeline

This notebook implements the preprocessing pipeline used for all baseline and
future models. All transformations are fit exclusively on the training data
to prevent information leakage.

Preprocessing Pipeline Design
We define the Target and Feature Set (Explicitly)

Target (y) -> label
Features (X) -> All columns except: id, attack_cat, label
Encoding Strategy (Important Decision)

We choose defensible, simple, and standard.

Chosen strategy: One-Hot Encoding

Why?

Transparent
No ordinal assumptions
Widely accepted in IDS literature
Works well with tree-based models
Categorical features were encoded using one-hot encoding to preserve nominal relationships without imposing artificial ordering.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/UNSW-NB15 Dataset/UNSW-NB15 dataset/CSV Files/Training and Testing Sets")

train = pd.read_csv(DATA_DIR / "UNSW_NB15_training-set.csv")
test  = pd.read_csv(DATA_DIR / "UNSW_NB15_testing-set.csv")

print("Training samples:", train.shape)
print("Testing samples :", test.shape)

assert train.shape[0] == 175341, "Training set size mismatch"
assert test.shape[0]  == 82332,  "Testing set size mismatch"

print("Correct UNSW-NB15 split confirmed")

Training samples: (175341, 45)
Testing samples : (82332, 45)
Correct UNSW-NB15 split confirmed


Drop non-feature columns

In [2]:
DROP_COLS = ['id', 'attack_cat']

X_train = train.drop(columns=DROP_COLS + ['label'])
y_train = train['label']

X_test  = test.drop(columns=DROP_COLS + ['label'])
y_test  = test['label']

print(X_train.shape, X_test.shape)

(175341, 42) (82332, 42)


Identify categorical vs numerical features

In [3]:
categorical_cols = X_train.select_dtypes(include="object").columns
print("Categorical columns:", categorical_cols.tolist())


Categorical columns: ['proto', 'service', 'state']


In [4]:
categorical_cols = ['proto', 'service', 'state']
numerical_cols = [col for col in X_train.columns if col not in categorical_cols]

print("Categorical:", categorical_cols)
print("Numerical:", len(numerical_cols))

Categorical: ['proto', 'service', 'state']
Numerical: 39


One-Hot Encode (train-fit, test-transform)

In [5]:
X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols)
X_test_encoded  = pd.get_dummies(X_test, columns=categorical_cols)

In [6]:
X_train_encoded = pd.get_dummies(
    X_train,
    columns=categorical_cols,
    drop_first=False
)

X_test_encoded = pd.get_dummies(
    X_test,
    columns=categorical_cols,
    drop_first=False
)

Align train and test features (NO leakage)

In [7]:
X_train_encoded, X_test_encoded = X_train_encoded.align(
    X_test_encoded,
    join="left",
    axis=1,
    fill_value=0
)


print(X_train_encoded.shape)
print(X_test_encoded.shape)

(175341, 194)
(82332, 194)


Final sanity checks

In [8]:
print(X_train_encoded.dtypes.value_counts())
assert not any(X_train_encoded.dtypes == "object")
assert not any(X_test_encoded.dtypes == "object")


bool       155
int64       28
float64     11
Name: count, dtype: int64


In [9]:
assert X_train_encoded.isnull().sum().sum() == 0
assert X_test_encoded.isnull().sum().sum() == 0

assert X_train.shape[1] == X_test.shape[1]
assert not X_train.isnull().any().any()
assert not X_test.isnull().any().any()

print("No missing values. Preprocessing complete.")

No missing values. Preprocessing complete.


Save processed data

In [ ]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

X_train_encoded.to_csv(PROCESSED_DIR / "X_train.csv", index=False)
X_test_encoded.to_csv(PROCESSED_DIR / "X_test.csv", index=False)
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=False)

print("Encoded datasets saved successfully")


Encoded datasets saved successfully


### Preprocessing Completion

The preprocessing pipeline produces clean, leakage‑free feature matrices for
model training and evaluation. These processed datasets are reused consistently
across all experimental phases.